# Caderno 4. Gera qrels a partir das anotações feitas por LLM

## 1. Parâmetros

In [1]:
PASTA_ANOTACOES_LLM = './dados/outputs/2 - llm evals/'
PASTA_QRELS_LLM = './dados/outputs/4 - qrel llm/'

## 2. Converter anotações para deixar num formato mais simples

In [8]:
import os
import re

def sanitize(text):
    text = str(text)
    text = re.sub(r"[^\w\-]+", "_", text)
    return text.strip("_")
    
def gerar_qrels(df):
    # ====== Agrupar por MODELO e PROMPT ======
    for (modelo, prompt), group in df.groupby(["MODELO", "PROMPT"]):    
        # Copiar dados relevantes
        df_out = group[["QUERY_KEY", "DOC_KEY", "AVALIACAO"]].copy()
    
        # Converter AVALIACAO para SCORE
        df_out = df_out.rename(columns={"AVALIACAO": "SCORE"})
    
        # Ordenar por QUERY_KEY e SCORE (maior score primeiro)
        df_out = df_out.sort_values(
            by=["QUERY_KEY", "SCORE"],
            ascending=[True, False]
        )
    
        # Gerar RANK por QUERY_KEY
        df_out["RANK"] = (
            df_out.groupby("QUERY_KEY")["SCORE"]
            .rank(method="first", ascending=False)
            .astype(int)
        )
    
        # Reordenar colunas
        df_out = df_out[["QUERY_KEY", "DOC_KEY", "SCORE", "RANK"]]
    
        # Criar nome do arquivo
        filename = f"{PASTA_QRELS_LLM}qrel_{sanitize(modelo)}_{sanitize(prompt)}.csv"
    
        # Salvar CSV
        df_out.to_csv(filename, index=False)
    
        print(f"Arquivo gerado: {filename}")


In [9]:
import pandas as pd

gerar_qrels(pd.read_csv(f"{PASTA_ANOTACOES_LLM}raw_llm_eval_deepseek.csv"))
gerar_qrels(pd.read_csv(f"{PASTA_ANOTACOES_LLM}raw_llm_eval_gpt-5-mini.csv"))
gerar_qrels(pd.read_csv(f"{PASTA_ANOTACOES_LLM}raw_llm_eval_sabiazinho.csv"))

Arquivo gerado: ./dados/outputs/3 - qrel llm/qrel_deepseek-chat_cot.csv
Arquivo gerado: ./dados/outputs/3 - qrel llm/qrel_deepseek-chat_simple.csv
Arquivo gerado: ./dados/outputs/3 - qrel llm/qrel_gpt-5-mini-2025-08-07_cot.csv
Arquivo gerado: ./dados/outputs/3 - qrel llm/qrel_gpt-5-mini-2025-08-07_simple.csv
Arquivo gerado: ./dados/outputs/3 - qrel llm/qrel_sabiazinho-4-2026-01-06_cot.csv
Arquivo gerado: ./dados/outputs/3 - qrel llm/qrel_sabiazinho-4-2026-01-06_simple.csv
